# Proyecto Final: Predicción de Ventas — Corporación Favorita
## Aplicación de Machine Learning a un Caso Real

**Autor:** Lore
**Dataset:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting) (Corporación Favorita, Ecuador)

### Contexto de negocio

Corporación Favorita es una de las cadenas de supermercados más grandes de Ecuador. El objetivo de este proyecto es predecir las ventas diarias de miles de productos (agrupados por familia) en decenas de tiendas, usando información histórica de ventas, promociones, precio del petróleo (variable clave en la economía ecuatoriana) y feriados/eventos especiales.

Una predicción de ventas precisa permite optimizar inventario, evitar quiebres de stock, reducir desperdicio y mejorar la planificación de personal y logística.

### Objetivo del proyecto

1. Explorar y entender los patrones de venta (estacionalidad, efecto de feriados, promociones, precio del petróleo).
2. Construir variables (features) relevantes a partir de los datos crudos.
3. Entrenar y comparar varios modelos de predicción (baseline, lineal, boosting).
4. Evaluar el desempeño con la métrica oficial de la competencia (RMSLE).
5. Interpretar el modelo y traducir los hallazgos en recomendaciones de negocio.


## 1. Descripción del dataset

El dataset viene dividido en varios archivos CSV que hay que combinar:

| Archivo | Contenido |
|---|---|
| `train.csv` | Ventas diarias históricas por tienda y familia de producto (`date`, `store_nbr`, `family`, `sales`, `onpromotion`) |
| `test.csv` | Mismas columnas que train, sin `sales` (a predecir) |
| `stores.csv` | Metadata de cada tienda (`store_nbr`, `city`, `state`, `type`, `cluster`) |
| `oil.csv` | Precio diario del petróleo WTI (`date`, `dcoilwtico`) |
| `holidays_events.csv` | Feriados y eventos especiales (`date`, `type`, `locale`, `locale_name`, `description`, `transferred`) |
| `transactions.csv` | Número de transacciones por tienda y día (`date`, `store_nbr`, `transactions`) |

**Cómo descargar los datos:**

1. Ve a la [página de la competencia en Kaggle](https://www.kaggle.com/competitions/store-sales-time-series-forecasting) y acepta las reglas (es gratis y necesario para descargar).
2. Descarga el ZIP y descomprímelo en una carpeta `data/` al lado de este notebook.
3. O usa la API de Kaggle desde la terminal:
   ```bash
   pip install kaggle
   kaggle competitions download -c store-sales-time-series-forecasting -p data/
   unzip data/store-sales-time-series-forecasting.zip -d data/
   ```


In [ ]:
# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_log_error, mean_absolute_error

# Modelo de boosting (instalar si hace falta: pip install xgboost)
from xgboost import XGBRegressor

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

DATA_DIR = "data/"


In [ ]:
# Carga de datos
train = pd.read_csv(DATA_DIR + "train.csv", parse_dates=["date"])
test = pd.read_csv(DATA_DIR + "test.csv", parse_dates=["date"])
stores = pd.read_csv(DATA_DIR + "stores.csv")
oil = pd.read_csv(DATA_DIR + "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(DATA_DIR + "holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(DATA_DIR + "transactions.csv", parse_dates=["date"])

print("train:", train.shape)
print("test:", test.shape)
print("stores:", stores.shape)
print("oil:", oil.shape)
print("holidays:", holidays.shape)
print("transactions:", transactions.shape)

train.head()


## 2. Análisis Exploratorio de Datos (EDA)

Antes de modelar, entendamos los patrones generales de venta: tendencia, estacionalidad, y el efecto de las variables externas.


In [ ]:
# Info general y nulos
print(train.info())
print("\nValores nulos por columna:")
print(train.isnull().sum())
print("\nRango de fechas:", train["date"].min(), "-", train["date"].max())
print("Tiendas únicas:", train["store_nbr"].nunique())
print("Familias de producto:", train["family"].nunique())


In [ ]:
# Tendencia de ventas totales en el tiempo
ventas_diarias = train.groupby("date")["sales"].sum().reset_index()

plt.plot(ventas_diarias["date"], ventas_diarias["sales"])
plt.title("Ventas totales diarias — Corporación Favorita")
plt.xlabel("Fecha")
plt.ylabel("Ventas totales")
plt.show()


In [ ]:
# Ventas por familia de producto (top 10)
top_familias = train.groupby("family")["sales"].sum().sort_values(ascending=False).head(10)

sns.barplot(x=top_familias.values, y=top_familias.index, palette="viridis")
plt.title("Top 10 familias de producto por ventas totales")
plt.xlabel("Ventas totales")
plt.show()


In [ ]:
# Ventas por tipo y cluster de tienda
ventas_tienda = train.merge(stores, on="store_nbr")
ventas_por_tipo = ventas_tienda.groupby("type")["sales"].mean().sort_values(ascending=False)

sns.barplot(x=ventas_por_tipo.index, y=ventas_por_tipo.values, palette="mako")
plt.title("Venta promedio por tipo de tienda")
plt.xlabel("Tipo de tienda")
plt.ylabel("Venta promedio")
plt.show()


In [ ]:
# Efecto del precio del petróleo en las ventas
oil_filled = oil.set_index("date").reindex(pd.date_range(train["date"].min(), train["date"].max()))
oil_filled["dcoilwtico"] = oil_filled["dcoilwtico"].interpolate()
oil_filled = oil_filled.reset_index().rename(columns={"index": "date"})

ventas_oil = ventas_diarias.merge(oil_filled, on="date", how="left")
correlacion = ventas_oil[["sales", "dcoilwtico"]].corr().iloc[0, 1]
print(f"Correlación ventas vs. precio del petróleo: {correlacion:.3f}")

fig, ax1 = plt.subplots()
ax1.plot(ventas_oil["date"], ventas_oil["sales"], color="tab:blue", label="Ventas")
ax1.set_ylabel("Ventas totales", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(ventas_oil["date"], ventas_oil["dcoilwtico"], color="tab:red", alpha=0.6, label="Precio petróleo")
ax2.set_ylabel("Precio del petróleo (WTI)", color="tab:red")
plt.title("Ventas totales vs. precio del petróleo")
plt.show()


In [ ]:
# Efecto de feriados en las ventas
holidays_nacionales = holidays[(holidays["locale"] == "National") & (~holidays["transferred"])]
fechas_feriado = set(holidays_nacionales["date"])

ventas_diarias["es_feriado"] = ventas_diarias["date"].isin(fechas_feriado)

sns.boxplot(data=ventas_diarias, x="es_feriado", y="sales", palette="Set2")
plt.title("Distribución de ventas: días normales vs. feriados nacionales")
plt.xlabel("¿Es feriado nacional?")
plt.show()

print(ventas_diarias.groupby("es_feriado")["sales"].mean())


In [ ]:
# Efecto de las promociones en las ventas
promo_vs_ventas = train.groupby("onpromotion")["sales"].mean().reset_index()
correlacion_promo = train[["sales", "onpromotion"]].corr().iloc[0, 1]
print(f"Correlación ventas vs. número de productos en promoción: {correlacion_promo:.3f}")


## 3. Ingeniería de variables (Feature Engineering)

Construimos variables derivadas de la fecha, combinamos las tablas externas (petróleo, feriados, tiendas, transacciones), y creamos variables de rezago (*lags*) y medias móviles, que suelen ser las más predictivas en series de tiempo de ventas.


In [ ]:
def crear_features_fecha(df):
    df = df.copy()
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
    df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
    df["is_month_end"] = df["date"].dt.is_month_end.astype(int)
    return df

train_fe = crear_features_fecha(train)
test_fe = crear_features_fecha(test)


In [ ]:
def combinar_datos_externos(df):
    df = df.merge(stores, on="store_nbr", how="left")
    df = df.merge(oil_filled, on="date", how="left")
    df["dcoilwtico"] = df["dcoilwtico"].fillna(method="ffill").fillna(method="bfill")
    df["es_feriado_nacional"] = df["date"].isin(fechas_feriado).astype(int)
    df = df.merge(transactions, on=["date", "store_nbr"], how="left")
    return df

train_fe = combinar_datos_externos(train_fe)
test_fe = combinar_datos_externos(test_fe)

train_fe.head()


In [ ]:
# Variables de rezago y medias móviles por tienda-familia
train_fe = train_fe.sort_values(["store_nbr", "family", "date"])

for lag in [7, 14, 28]:
    train_fe[f"sales_lag_{lag}"] = train_fe.groupby(["store_nbr", "family"])["sales"].shift(lag)

for window in [7, 28]:
    train_fe[f"sales_rolling_mean_{window}"] = (
        train_fe.groupby(["store_nbr", "family"])["sales"]
        .shift(1)
        .rolling(window)
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

train_fe = train_fe.dropna(subset=["sales_lag_28"]).reset_index(drop=True)
train_fe.head()


In [ ]:
# Codificación de variables categóricas
categoricas = ["family", "city", "state", "type", "cluster"]
train_fe = pd.get_dummies(train_fe, columns=categoricas, drop_first=True)


## 4. División train / validación

En series de tiempo **no** se hace un split aleatorio: hay que respetar el orden cronológico para no filtrar información del futuro (*data leakage*). Usamos los últimos ~15% de fechas como validación.


In [ ]:
fecha_corte = train_fe["date"].quantile(0.85)

features_excluir = ["id", "date", "sales", "transactions"]
features = [c for c in train_fe.columns if c not in features_excluir]

X_train = train_fe[train_fe["date"] <= fecha_corte][features]
y_train = train_fe[train_fe["date"] <= fecha_corte]["sales"]
X_val = train_fe[train_fe["date"] > fecha_corte][features]
y_val = train_fe[train_fe["date"] > fecha_corte]["sales"]

print("Train:", X_train.shape, "Validación:", X_val.shape)


## 5. Modelado

Entrenamos y comparamos tres enfoques, de menor a mayor complejidad:

1. **Baseline**: media móvil de los últimos 7 días (sin aprendizaje).
2. **Regresión lineal**: modelo simple e interpretable.
3. **XGBoost**: modelo de gradient boosting, generalmente el de mejor desempeño en este tipo de problema tabular.

La métrica oficial de la competencia es **RMSLE** (Root Mean Squared Logarithmic Error), que penaliza más los errores relativos y maneja bien la asimetría de las ventas.


In [ ]:
def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)  # las ventas no pueden ser negativas
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

resultados = {}


In [ ]:
# Baseline: usar la media móvil de 7 días como predicción
pred_baseline = X_val["sales_rolling_mean_7"].fillna(0)
resultados["Baseline (media móvil 7d)"] = rmsle(y_val, pred_baseline)
print(f"RMSLE Baseline: {resultados['Baseline (media móvil 7d)']:.4f}")


In [ ]:
# Regresión lineal
lr = LinearRegression()
lr.fit(X_train.fillna(0), y_train)
pred_lr = lr.predict(X_val.fillna(0))
resultados["Regresión Lineal"] = rmsle(y_val, pred_lr)
print(f"RMSLE Regresión Lineal: {resultados['Regresión Lineal']:.4f}")


In [ ]:
# XGBoost
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train.fillna(0), y_train)
pred_xgb = xgb_model.predict(X_val.fillna(0))
resultados["XGBoost"] = rmsle(y_val, pred_xgb)
print(f"RMSLE XGBoost: {resultados['XGBoost']:.4f}")


In [ ]:
# Comparación de modelos
tabla_resultados = pd.DataFrame(resultados.items(), columns=["Modelo", "RMSLE"]).sort_values("RMSLE")
print(tabla_resultados)

sns.barplot(data=tabla_resultados, x="RMSLE", y="Modelo", palette="crest")
plt.title("Comparación de modelos — RMSLE en validación (menor es mejor)")
plt.show()


## 6. Interpretación del modelo

¿Qué variables son las más importantes para predecir ventas? Esto conecta el modelo con insights accionables para el negocio.


In [ ]:
importancias = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
top_importancias = importancias.sort_values(ascending=False).head(15)

sns.barplot(x=top_importancias.values, y=top_importancias.index, palette="flare")
plt.title("Top 15 variables más importantes (XGBoost)")
plt.xlabel("Importancia")
plt.show()


## 7. Conclusiones y recomendaciones de negocio

*(Completar con los resultados reales una vez ejecutado el notebook con los datos)*

- **Modelo ganador**: el modelo con menor RMSLE en validación, indicando que captura mejor los patrones de venta.
- **Estacionalidad**: identificar qué días de la semana / meses concentran más ventas para ajustar inventario y personal.
- **Petróleo**: cuantificar cuánto varían las ventas ante cambios en el precio del petróleo (relevante para la economía ecuatoriana).
- **Feriados**: dimensionar el incremento de ventas en feriados nacionales para anticipar stock.
- **Promociones**: estimar el "levantamiento" (uplift) real que generan las promociones sobre las ventas base.
- **Variables más importantes**: usar el ranking de importancia para priorizar qué datos recolectar/mejorar a futuro (ej. transacciones, rezagos recientes).

### Próximos pasos (opcional, para ir más allá)

- Afinar hiperparámetros del modelo con `GridSearchCV` u `Optuna`.
- Probar modelos específicos de series de tiempo (Prophet, SARIMA) y comparar contra XGBoost.
- Entrenar un modelo por familia de producto o por cluster de tienda (modelos especializados).
- Construir un dashboard interactivo (Streamlit) para explorar predicciones por tienda y familia.
- Generar el archivo `submission.csv` y subirlo a la competencia de Kaggle para obtener una puntuación pública.
